In [1]:
import json
import dlt
from dlt.sources.helpers import requests   # cliente con reintentos incluido

BASE = "https://dadesobertes.gva.es/api/3/action/datastore_search"
RECURSO = "0f06c6db-ec58-4ac4-9913-5f91022ce829"   # medidas por días, últimos registros
ESTACION = "03065007"                              # Elx - Parc de Bombers

In [2]:
@dlt.resource(
    table_name="mediciones",
    primary_key=["cd_estacion", "fecha", "identificador"],
    write_disposition="merge",
)
def mediciones(fecha=dlt.sources.incremental("fecha", initial_value="2026-01-01")):
    offset = 0
    while True:
        r = requests.get(BASE, params={
            "resource_id": RECURSO,
            "filters": json.dumps({"cd_estacion": ESTACION}),
            "sort": "_id asc",
            "limit": 100,
            "offset": offset,
        })
        pagina = r.json()["result"]["records"]
        if not pagina:
            break
        yield pagina
        offset += len(pagina)
        print(offset)

In [5]:
pipeline = dlt.pipeline(
    pipeline_name="rvvcca",
    destination="filesystem",
    dataset_name="rvvcca",
)

In [6]:
info = pipeline.run(mediciones, loader_file_format="parquet")
print("cursor:", pipeline.state["sources"]["rvvcca"]["resources"]["mediciones"]["incremental"]["fecha"]["last_value"])
print("filas cargadas:", pipeline.last_trace.last_normalize_info.row_counts.get("mediciones", 0))
print("paquetes de carga:", len(info.load_packages))

36


2026-08-29 09:58:54,904|[WARNING]|503|126378431234880|dlt|filesystem.py|prepare_load_table:958|Falling back to `append` on `mediciones`.
2026-08-29 09:58:54,906|[WARNING]|503|126378431234880|dlt|filesystem.py|prepare_load_table:958|Falling back to `append` on `mediciones`.
2026-08-29 09:58:54,954|[WARNING]|503|126378431234880|dlt|filesystem.py|prepare_load_table:958|Falling back to `append` on `mediciones`.


cursor: 27/08/2026
filas cargadas: 36
paquetes de carga: 1
